In [ ]:
import pandas as pd
import requests
import re
import zipfile
import io

def task_1_unique_service_list(metadata_url):
    # 1. LIVE DATA DISCOVERY: Extracting current hospital list and URLs
    try:
        response = requests.get(metadata_url, timeout=20)
        # Regex to capture the specific hospital name and its data link
        hospital_matches = re.findall(r"location-name:\s*(.*)\n(?:.*\n)?mrf-url:\s*(.*)", response.text)
    except Exception as e:
        print(f"Connection Error: {e}")
        return

    results = []

    # 2. DATA PROCESSING: Iterating through each hospital to count services
    for name, url in hospital_matches:
        name, url = name.strip(), url.strip()
        print(f"Analyzing: {name}")
        
        try:
            r = requests.get(url, timeout=120)
            # 2026 schema usually bundles data in ZIPs; we extract the CSV in memory
            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                csv_file = [f for f in z.namelist() if f.endswith('.csv')][0]
                with z.open(csv_file) as f:
                    # skip 2 rows to bypass CMS headers and reach the data
                    df = pd.read_csv(f, skiprows=2, low_memory=False, encoding_errors='replace')
            
            # Standardize columns to lower-case to ensure 'code_2' is found
            df.columns = [str(c).replace('|', '_').lower().strip() for c in df.columns]
            
            # TASK 1 CORE: Calculating the count of unique clinical service codes
            unique_count = df['code_2'].nunique()
            results.append({"Hospital": name, "Unique_Services": unique_count})
            
        except Exception as e:
            print(f"Skipped {name}: Data format error.")

    # 3. FINAL OUTPUT: Sorting and Saving
    if results:
        final_list = pd.DataFrame(results).sort_values(by="Unique_Services", ascending=False)
        final_list.to_csv("task_1_hospital_services.csv", index=False)
        
        print("\n--- TASK 1 RESULT: UNIQUE SERVICES PER HOSPITAL ---")
        print(final_list.to_string(index=False))
        return final_list

# Execute using MGB's live metadata
MGB_ROOT = "https://www.massgeneralbrigham.org/cms-hpt.txt"
task_1_unique_service_list(MGB_ROOT)